# TP 4 — Spark SQL : requêter le fil rouge e-commerce

**Big Data Engineering — Master 1 — DMI/FST/UCAD — Prof. Samba Ndiaye**

## Consignes
- Complétez toutes les cellules marquées `# === À COMPLÉTER ===` (remplacez les `...`).
- Rédigez vos réponses dans les cellules *Votre réponse :*.
- Le notebook doit s'exécuter **de bout en bout** (Kernel > Restart & Run All) avant d'être poussé.
- Livrable : `notebooks/TP4_spark_sql.ipynb` **avec les sorties visibles**, poussé sur votre dépôt avant la séance 5.

## Déroulé
| Partie | Contenu | Durée |
|---|---|---|
| A | Mise en place : données, SparkSession, vues | 15 min |
| B | Premières requêtes SQL | 25 min |
| C | L'enquête FCFA | 30 min |
| D | Les indicateurs de la direction | 40 min |
| E | SQL ou API ? `explain()` tranche | 20 min |
| F | Discussion et quiz | 20 min |

## 0. Vérification de l'environnement

Données : si le dossier `data/` est absent, exécutez d'abord dans un terminal (ou une cellule `!`) :
```
python generate_data.py --scale 0.1 --outdir data
```
Graine 42 : tous les étudiants ont **exactement** les mêmes données.

In [1]:
import sys
print("Python :", sys.version.split()[0])

from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .appName("TP4-SparkSQL")
         .master("local[*]")
         .getOrCreate())
print("Spark  :", spark.version)
spark.sparkContext.setLogLevel("ERROR")

Python : 3.12.0
Spark  : 4.2.0


### Tableau de relevés

Il se remplit **au fil du TP** ; la dernière cellule du notebook l'affiche. Un notebook sans chiffres n'est pas un livrable.

In [2]:
releves = {
    "A_nb_lignes_clients":        None,
    "A_nb_lignes_commandes":      None,
    "A_type_montant_total_fcfa":  None,   # ex. "string"
    "C2_nb_valeurs_polluees":     None,
    "C1_ca_naif":                 None,
    "C4_ca_nettoye":              None,
    "C5_ecart_fcfa":              None,
    "C5_ecart_pct":               None,
    "D1_part_ca_livree_pct":      None,
    "D2_mois_record":             None,
    "D3_panier_moyen_mobile":     None,
    "D5_part_mobile_money_pct":   None,
}

## Partie A — Mise en place (15 min)

### A.1 — Charger les quatre sources et créer les vues

Chargez `customers.csv`, `orders.csv`, `products.csv` (CSV : `header=True`, `inferSchema=True`) et `payments.json`, puis créez les vues temporaires `clients`, `commandes`, `produits`, `paiements`.

In [5]:
base = "../data/"

# === Chargement des données ===
clients = spark.read.option("header", True).option("inferSchema", True).csv(base + "customers.csv")
commandes = spark.read.option("header", True).option("inferSchema", True).csv(base + "orders.csv")
produits = spark.read.option("header", True).option("inferSchema", True).csv(base + "products.csv")
paiements = spark.read.json(base + "payments.json")

# === Création des vues temporaires ===
clients.createOrReplaceTempView("clients")
commandes.createOrReplaceTempView("commandes")
produits.createOrReplaceTempView("produits")
paiements.createOrReplaceTempView("paiements")

# === Vérification ===
print("📋 Tables disponibles :")
spark.catalog.listTables()

# === Afficher les schémas ===
print("\n📊 Schéma de 'clients' :")
clients.printSchema()

print("\n📊 Schéma de 'commandes' :")
commandes.printSchema()

print("\n📊 Schéma de 'produits' :")
produits.printSchema()

print("\n📊 Schéma de 'paiements' :")
paiements.printSchema()

# === Compter les lignes ===
print("\n📈 Nombre de lignes :")
print(f"  Clients   : {clients.count():,}")
print(f"  Commandes : {commandes.count():,}")
print(f"  Produits  : {produits.count():,}")
print(f"  Paiements : {paiements.count():,}")

📋 Tables disponibles :

📊 Schéma de 'clients' :
root
 |-- customer_id: string (nullable = true)
 |-- prenom: string (nullable = true)
 |-- nom: string (nullable = true)
 |-- email: string (nullable = true)
 |-- telephone: string (nullable = true)
 |-- adresse: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- region: string (nullable = true)
 |-- date_naissance: date (nullable = true)
 |-- date_inscription: date (nullable = true)


📊 Schéma de 'commandes' :
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- date_commande: timestamp (nullable = true)
 |-- statut: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- frais_livraison_fcfa: integer (nullable = true)
 |-- montant_total_fcfa: string (nullable = true)


📊 Schéma de 'produits' :
root
 |-- product_id: string (nullable = true)
 |-- nom_produit: string (nullable = true)
 |-- categorie: string (nullable = true)
 |-- marque: string (nullable = true)
 |-- prix_u

### A.2 — Premier relevé

Comptez les lignes de `clients` et `commandes`, affichez le schéma de `commandes`, et relevez le **type inféré** de `montant_total_fcfa` et de `frais_livraison_fcfa`.

In [8]:
# === À COMPLÉTER ===
releves["A_nb_lignes_clients"]   = spark.sql("SELECT COUNT(*) AS n FROM clients").first()["n"]
releves["A_nb_lignes_commandes"] = ...

commandes.printSchema()
releves["A_type_montant_total_fcfa"] = ...   # recopiez le type lu dans le schema
print(releves)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- date_commande: timestamp (nullable = true)
 |-- statut: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- frais_livraison_fcfa: integer (nullable = true)
 |-- montant_total_fcfa: string (nullable = true)

{'A_nb_lignes_clients': 50250, 'A_nb_lignes_commandes': Ellipsis, 'A_type_montant_total_fcfa': Ellipsis, 'C2_nb_valeurs_polluees': None, 'C1_ca_naif': None, 'C4_ca_nettoye': None, 'C5_ecart_fcfa': None, 'C5_ecart_pct': None, 'D1_part_ca_livree_pct': None, 'D2_mois_record': None, 'D3_panier_moyen_mobile': None, 'D5_part_mobile_money_pct': None}


**Question A** — Une des deux colonnes de montants n'a pas le type attendu. Laquelle, et qu'en déduisez-vous sur le contenu du fichier ? (Vous vérifierez votre hypothèse en partie C.)

*Votre réponse :*

…

## Partie B — Premières requêtes SQL (25 min)

Une requête par cellule, résultat affiché avec `.show()`.

### B1 — Les 10 premiers clients de Dakar
Colonnes : `customer_id`, `prenom`, `nom`, `ville`.

In [11]:
spark.sql("""
    SELECT customer_id, prenom, nom, ville 
    FROM clients 
    WHERE ville = 'Dakar' 
    LIMIT 10
""").show()

+-----------+----------+-----+-----+
|customer_id|    prenom|  nom|ville|
+-----------+----------+-----+-----+
|    C032075|     Ndèye|Thiam|Dakar|
|    C004690|      Omar| Kébé|Dakar|
|    C013308|     Adama| Fall|Dakar|
|    C036810|    Moussa|   Lô|Dakar|
|    C012515|  Maguette| Sène|Dakar|
|    C011077|  Dieynaba| Sané|Dakar|
|    C011730|Souleymane| Faye|Dakar|
|    C014742|     Ndèye|Guèye|Dakar|
|    C038399|    Diarra|   Ba|Dakar|
|    C032604|      Omar|Diouf|Dakar|
+-----------+----------+-----+-----+



### B2 — Combien de villes distinctes dans `clients` ?

In [12]:
spark.sql("""
    SELECT COUNT(DISTINCT ville) AS nb_villes_distinctes 
    FROM clients
""").show()

+--------------------+
|nb_villes_distinctes|
+--------------------+
|                  65|
+--------------------+



### B3 — Top 10 des produits les plus chers
Nom et prix, tri décroissant.

In [13]:
spark.sql("""
    SELECT nom_produit, prix_unitaire_fcfa 
    FROM produits 
    ORDER BY prix_unitaire_fcfa DESC 
    LIMIT 10
""").show()

+--------------------+------------------+
|         nom_produit|prix_unitaire_fcfa|
+--------------------+------------------+
|Dakar Style Infor...|            885500|
|Dakar Style Infor...|            874000|
|Hisense Informati...|            866000|
|Infinix Informati...|            859000|
|Royal Informatiqu...|            853500|
|Itel Informatique...|            843000|
|Lenovo Informatiq...|            803000|
|LG Informatique 0824|            801000|
|Dakar Style Infor...|            777500|
|Penc Mi Informati...|            755000|
+--------------------+------------------+



### B4 — Commandes livrées, canal mobile, décembre 2025
Combien de commandes `livree` du canal `mobile_app` en décembre 2025 ?

In [14]:
spark.sql("""
    SELECT COUNT(*) AS nb_commandes_livrees_mobile_dec2025 
    FROM commandes 
    WHERE canal = 'mobile_app' 
      AND statut = 'livrée' 
      AND date_commande BETWEEN '2025-12-01' AND '2025-12-31'
""").show()

+-----------------------------------+
|nb_commandes_livrees_mobile_dec2025|
+-----------------------------------+
|                              16415|
+-----------------------------------+



### B5 — Emails manquants
Combien de clients ont un email `NULL` **ou** égal à `'N/A'` ? (Rappel séance 3 : le manquant a deux visages — et `= NULL` ne fonctionne pas.)

In [15]:
spark.sql("""
    SELECT COUNT(*) AS nb_clients_email_manquant 
    FROM clients 
    WHERE email = '' OR email = 'N/A' OR email IS NULL
""").show()

+-------------------------+
|nb_clients_email_manquant|
+-------------------------+
|                     1506|
+-------------------------+



## Partie C — L'enquête FCFA (30 min)

### C1 — Le symptôme : la somme naïve
Calculez le CA total directement sur la colonne brute, et **notez le résultat**.

In [18]:
# === C1 - CA naïf avec nettoyage préalable ===
print("=" * 60)
print("C1 - CA total (naïf, avec nettoyage pour éviter l'erreur)")
print("=" * 60)

# Version 1 : Utiliser TRY_CAST (Spark 3.3+)
ca_naif = spark.sql("""
    SELECT SUM(TRY_CAST(montant_total_fcfa AS BIGINT)) AS ca 
    FROM commandes
""").first()["ca"]

releves["C1_ca_naif"] = ca_naif
print(f"CA naif : {ca_naif:,.0f} FCFA")

C1 - CA total (naïf, avec nettoyage pour éviter l'erreur)
CA naif : 86,558,339,900 FCFA


### C2 — Diagnostiquer
Comptez les valeurs de `montant_total_fcfa` qui ne sont **pas** de purs nombres, puis affichez 10 valeurs fautives distinctes.

In [19]:
# === C2 - Compter les lignes polluées ===
print("=" * 60)
print("C2 - Lignes avec montant non numérique")
print("=" * 60)

nb_pollues = spark.sql("""
    SELECT COUNT(*) AS nb
    FROM commandes
    WHERE montant_total_fcfa NOT RLIKE '^[0-9]+$'
""").first()["nb"]
releves["C2_nb_valeurs_polluees"] = nb_pollues
print(f"Valeurs polluées : {nb_pollues:,}")

# Afficher les 10 valeurs fautives distinctes
print("\nÉchantillon des 10 valeurs fautives distinctes :")
spark.sql("""
    SELECT DISTINCT montant_total_fcfa AS valeurs_fautives
    FROM commandes
    WHERE montant_total_fcfa NOT RLIKE '^[0-9]+$'
    LIMIT 10
""").show(truncate=False)

# Motif de pollution observé
motif_pollution = "Valeurs avec suffixe ' FCFA' et/ou espaces parasites"
releves["C2_motif_pollution"] = motif_pollution
print(f"\nMotif de pollution observé : {motif_pollution}")

C2 - Lignes avec montant non numérique
Valeurs polluées : 5,000

Échantillon des 10 valeurs fautives distinctes :
+----------------+
|valeurs_fautives|
+----------------+
|21000 FCFA      |
|281700 FCFA     |
|245500 FCFA     |
|73000 FCFA      |
|739500 FCFA     |
|114500 FCFA     |
|154000 FCFA     |
|128500 FCFA     |
|57000 FCFA      |
|138000 FCFA     |
+----------------+


Motif de pollution observé : Valeurs avec suffixe ' FCFA' et/ou espaces parasites


**Question C** — Expliquez en deux phrases pourquoi la requête C1 rend un résultat **faux sans lever d'erreur**.

*Votre réponse :*

…

### C3 — Nettoyer : la vue `commandes_clean`
Complétez la regex : supprimer **tout ce qui n'est pas un chiffre**, puis caster en `BIGINT`. On conserve la colonne brute sous `montant_raw`.

In [20]:
# === C3 - Création de la vue nettoyée ===
print("=" * 60)
print("C3 - Création de la vue commandes_clean")
print("=" * 60)

spark.sql("""
    CREATE OR REPLACE TEMP VIEW commandes_clean AS
    SELECT order_id, customer_id, date_commande, statut, canal,
           frais_livraison_fcfa,
           montant_total_fcfa AS montant_raw,
           CAST(REGEXP_REPLACE(TRIM(montant_total_fcfa), '[^0-9]', '') AS BIGINT) AS montant_fcfa
    FROM commandes
""")

print("✅ Vue 'commandes_clean' créée avec succès !")

# Vérification : afficher les 5 premières lignes
print("\nAperçu des données nettoyées :")
spark.sql("SELECT montant_raw, montant_fcfa FROM commandes_clean LIMIT 5").show(truncate=False)

C3 - Création de la vue commandes_clean
✅ Vue 'commandes_clean' créée avec succès !

Aperçu des données nettoyées :
+-----------+------------+
|montant_raw|montant_fcfa|
+-----------+------------+
|202700     |202700      |
|10000      |10000       |
|28000      |28000       |
|322000     |322000      |
|45000      |45000       |
+-----------+------------+



### C4 — Valider : mesurer, pas affirmer
Vérifiez qu'aucun `NULL` n'a été produit, contrôlez `MIN`/`MAX`, et recalculez le CA.

In [21]:
# === C4 - Validation du nettoyage ===
print("=" * 60)
print("C4 - Validation du nettoyage")
print("=" * 60)

validation = spark.sql("""
    SELECT COUNT(*)                        AS nb_lignes,
           COUNT(montant_fcfa)             AS nb_castes,
           COUNT(*) - COUNT(montant_fcfa)  AS nb_null,
           MIN(montant_fcfa)               AS mini,
           MAX(montant_fcfa)               AS maxi,
           SUM(montant_fcfa)               AS ca_total
    FROM commandes_clean
""")
validation.show()

# Extraction des valeurs pour le relevé
releves["C4_nb_lignes"] = validation.first()["nb_lignes"]
releves["C4_nb_castes"] = validation.first()["nb_castes"]
releves["C4_nb_null"] = validation.first()["nb_null"]
releves["C4_mini"] = validation.first()["mini"]
releves["C4_maxi"] = validation.first()["maxi"]
releves["C4_ca_nettoye"] = validation.first()["ca_total"]

print("\n" + "-" * 60)
print("📊 Relevé C4 :")
print(f"  Nombre de lignes          : {releves['C4_nb_lignes']:,}")
print(f"  Nombre de valeurs CASTées  : {releves['C4_nb_castes']:,}")
print(f"  Nombre de NULL            : {releves['C4_nb_null']} ✅")
print(f"  Montant MIN               : {releves['C4_mini']:,.0f} FCFA")
print(f"  Montant MAX               : {releves['C4_maxi']:,.0f} FCFA")
print(f"  CA total nettoyé          : {releves['C4_ca_nettoye']:,.0f} FCFA")
print("-" * 60)

# Vérification que nb_null = 0 (attendu)
if releves['C4_nb_null'] == 0:
    print("✅ Aucune valeur NULL détectée - Nettoyage réussi !")
else:
    print(f"⚠️ Attention : {releves['C4_nb_null']} valeurs NULL détectées")

C4 - Validation du nettoyage
+---------+---------+-------+----+-------+-----------+
|nb_lignes|nb_castes|nb_null|mini|   maxi|   ca_total|
+---------+---------+-------+----+-------+-----------+
|   500000|   500000|      0| 400|4431000|87451665200|
+---------+---------+-------+----+-------+-----------+


------------------------------------------------------------
📊 Relevé C4 :
  Nombre de lignes          : 500,000
  Nombre de valeurs CASTées  : 500,000
  Nombre de NULL            : 0 ✅
  Montant MIN               : 400 FCFA
  Montant MAX               : 4,431,000 FCFA
  CA total nettoyé          : 87,451,665,200 FCFA
------------------------------------------------------------
✅ Aucune valeur NULL détectée - Nettoyage réussi !


### C5 — La preuve chiffrée
Calculez l'écart entre le CA naïf (C1) et le CA nettoyé (C4), en FCFA et en pourcentage.

In [22]:
# === C5 - Version avec calcul direct depuis Spark (recommandée) ===
print("=" * 60)
print("C5 - Écart entre CA naïf et CA nettoyé (PREUVE)")
print("=" * 60)

# Méthode 1 : Utiliser les valeurs déjà dans releves
ecart = releves["C4_ca_nettoye"] - releves["C1_ca_naif"]
releves["C5_ecart_fcfa"] = ecart
if releves["C1_ca_naif"] != 0:
    releves["C5_ecart_pct"] = (ecart / releves["C1_ca_naif"]) * 100
else:
    releves["C5_ecart_pct"] = 0

print(f"📊 Écart calculé depuis les relevés :")
print(f"  CA naïf (C1)    : {releves['C1_ca_naif']:,.0f} FCFA")
print(f"  CA nettoyé (C4) : {releves['C4_ca_nettoye']:,.0f} FCFA")
print(f"  Écart absolu    : {releves['C5_ecart_fcfa']:,.0f} FCFA")
print(f"  Écart %         : {releves['C5_ecart_pct']:.2f}%")

# Méthode 2 : Vérification avec une requête Spark directe
print("\n" + "-" * 60)
print("Vérification directe avec Spark :")
print("-" * 60)

verification = spark.sql("""
    SELECT 
        SUM(CAST(REGEXP_REPLACE(TRIM(montant_total_fcfa), '[^0-9]', '') AS BIGINT)) AS ca_naif,
        SUM(montant_fcfa) AS ca_net,
        SUM(montant_fcfa) - SUM(CAST(REGEXP_REPLACE(TRIM(montant_total_fcfa), '[^0-9]', '') AS BIGINT)) AS ecart_fcfa,
        ROUND(
            (SUM(montant_fcfa) - SUM(CAST(REGEXP_REPLACE(TRIM(montant_total_fcfa), '[^0-9]', '') AS BIGINT))) 
            * 100.0 / SUM(CAST(REGEXP_REPLACE(TRIM(montant_total_fcfa), '[^0-9]', '') AS BIGINT)), 
            2
        ) AS ecart_pct
    FROM commandes_clean
""")
verification.show()

# Vérifier la cohérence
verif_row = verification.first()
releves["C5_verif_ca_naif"] = verif_row["ca_naif"]
releves["C5_verif_ca_net"] = verif_row["ca_net"]
releves["C5_verif_ecart_fcfa"] = verif_row["ecart_fcfa"]
releves["C5_verif_ecart_pct"] = verif_row["ecart_pct"]

print("\n" + "-" * 60)
print("🔍 Comparaison des résultats :")
print("-" * 60)
print(f"  Écart absolu (releves) : {releves['C5_ecart_fcfa']:,.0f} FCFA")
print(f"  Écart absolu (Spark)   : {releves['C5_verif_ecart_fcfa']:,.0f} FCFA")
print(f"  ✅ Cohérents" if abs(releves['C5_ecart_fcfa'] - releves['C5_verif_ecart_fcfa']) < 1 else "  ⚠️ Incohérents")

C5 - Écart entre CA naïf et CA nettoyé (PREUVE)
📊 Écart calculé depuis les relevés :
  CA naïf (C1)    : 86,558,339,900 FCFA
  CA nettoyé (C4) : 87,451,665,200 FCFA
  Écart absolu    : 893,325,300 FCFA
  Écart %         : 1.03%

------------------------------------------------------------
Vérification directe avec Spark :
------------------------------------------------------------


{"ts": "2026-08-07 20:28:18.535", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `montant_total_fcfa` cannot be resolved. Did you mean one of the following? [`montant_fcfa`, `montant_raw`, `canal`, `statut`, `customer_id`]. SQLSTATE: 42703", "context": {"errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o28.sql.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `montant_total_fcfa` cannot be resolved. Did you mean one of the following? [`montant_fcfa`, `montant_raw`, `canal`, `statut`, `customer_id`]. SQLSTATE: 42703; line 3 pos 37;\n'Aggregate ['SUM(cast('REGEXP_REPLACE('TRIM('montant_total_fcfa), [^0-9], ) as bigint)) AS ca_naif#447, sum(montant_fcfa#460L) AS ca_net#448L, (sum(montant_fcfa#460L) - 'SUM(cast('REGEXP_REPLACE('

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `montant_total_fcfa` cannot be resolved. Did you mean one of the following? [`montant_fcfa`, `montant_raw`, `canal`, `statut`, `customer_id`]. SQLSTATE: 42703; line 3 pos 37;
'Aggregate ['SUM(cast('REGEXP_REPLACE('TRIM('montant_total_fcfa), [^0-9], ) as bigint)) AS ca_naif#447, sum(montant_fcfa#460L) AS ca_net#448L, (sum(montant_fcfa#460L) - 'SUM(cast('REGEXP_REPLACE('TRIM('montant_total_fcfa), [^0-9], ) as bigint))) AS ecart_fcfa#449, 'ROUND((((sum(montant_fcfa#460L) - 'SUM(cast('REGEXP_REPLACE('TRIM('montant_total_fcfa), [^0-9], ) as bigint))) * 100.0) / 'SUM(cast('REGEXP_REPLACE('TRIM('montant_total_fcfa), [^0-9], ) as bigint))), 2) AS ecart_pct#450]
+- SubqueryAlias commandes_clean
   +- View (`commandes_clean`, [order_id#453, customer_id#454, date_commande#455, statut#456, canal#457, frais_livraison_fcfa#458, montant_raw#459, montant_fcfa#460L])
      +- Project [cast(order_i

## Partie D — Les indicateurs de la direction (40 min)

Toutes les requêtes portent sur `commandes_clean` (et `paiements` pour D5). Après chaque résultat, ajoutez **une phrase d'interprétation métier** dans la cellule markdown qui suit.

### D1 — CA et commandes par statut
Quelle part du CA est réellement `livree` ?

In [23]:
# === D1 - CA total et nombre de commandes par statut ===
print("=" * 60)
print("D1 - CA et commandes par statut")
print("=" * 60)

# Requête principale
spark.sql("""
    SELECT statut,
           COUNT(*)          AS nb_commandes,
           SUM(montant_fcfa) AS ca_fcfa
    FROM commandes_clean
    GROUP BY statut
    ORDER BY ca_fcfa DESC
""").show()

# Calcul de la part du CA livré
resultats_d1 = spark.sql("""
    SELECT 
        SUM(CASE WHEN statut = 'livrée' THEN montant_fcfa ELSE 0 END) AS ca_livre,
        SUM(montant_fcfa) AS ca_total,
        ROUND(SUM(CASE WHEN statut = 'livrée' THEN montant_fcfa ELSE 0 END) * 100.0 / SUM(montant_fcfa), 2) AS part_livre_pct
    FROM commandes_clean
""").first()

releves["D1_ca_livre"] = resultats_d1["ca_livre"]
releves["D1_ca_total"] = resultats_d1["ca_total"]
releves["D1_part_ca_livree_pct"] = resultats_d1["part_livre_pct"]

print("\n" + "-" * 60)
print("📊 RELEVÉ D1 - Part du CA livré :")
print(f"  CA total        : {releves['D1_ca_total']:,.0f} FCFA")
print(f"  CA livré        : {releves['D1_ca_livre']:,.0f} FCFA")
print(f"  Part du CA livré : {releves['D1_part_ca_livree_pct']:.2f}%")
print("-" * 60)

D1 - CA et commandes par statut
+---------+------------+-----------+
|   statut|nb_commandes|    ca_fcfa|
+---------+------------+-----------+
|   livrée|      389865|68111463600|
|  annulée|       44842| 7838725400|
| en_cours|       40237| 7085741500|
|retournée|       25056| 4415734700|
+---------+------------+-----------+


------------------------------------------------------------
📊 RELEVÉ D1 - Part du CA livré :
  CA total        : 87,451,665,200 FCFA
  CA livré        : 68,111,463,600 FCFA
  Part du CA livré : 77.88%
------------------------------------------------------------


*Votre réponse :*

…

### D2 — Le CA mensuel des commandes livrées
`date_trunc('month', ...)`, tri chronologique. Repérez la tendance et le mois record.

In [25]:
# === D2 - CA mensuel des commandes livrées ===
print("=" * 60)
print("D2 - CA mensuel des commandes livrées")
print("=" * 60)

ca_mensuel = spark.sql("""
    SELECT DATE_TRUNC('month', date_commande) AS mois,
           COUNT(*)                           AS nb_commandes,
           SUM(montant_fcfa)                  AS ca_fcfa
    FROM commandes_clean
    WHERE statut = 'livrée'
    GROUP BY DATE_TRUNC('month', date_commande)
    ORDER BY mois
""")
ca_mensuel.show(24, truncate=False)

# === Extraction du mois record ===
mois_record = spark.sql("""
    SELECT DATE_FORMAT(DATE_TRUNC('month', date_commande), 'yyyy-MM') AS mois,
           SUM(montant_fcfa) AS ca_fcfa
    FROM commandes_clean
    WHERE statut = 'livrée'
    GROUP BY DATE_TRUNC('month', date_commande)
    ORDER BY ca_fcfa DESC
    LIMIT 1
""").first()

releves["D2_mois_record"] = mois_record["mois"]
releves["D2_ca_mois_record"] = mois_record["ca_fcfa"]

print("\n" + "-" * 60)
print("📊 RELEVÉ D2 - Mois record :")
print(f"  Mois record : {releves['D2_mois_record']}")
print(f"  CA du mois  : {releves['D2_ca_mois_record']:,.0f} FCFA")
print("-" * 60)

D2 - CA mensuel des commandes livrées
+-------------------+------------+----------+
|mois               |nb_commandes|ca_fcfa   |
+-------------------+------------+----------+
|2024-07-01 00:00:00|12345       |2075417200|
|2024-08-01 00:00:00|12820       |2288823300|
|2024-09-01 00:00:00|12338       |2165224100|
|2024-10-01 00:00:00|13413       |2303534600|
|2024-11-01 00:00:00|13148       |2287422700|
|2024-12-01 00:00:00|20959       |3724236500|
|2025-01-01 00:00:00|14217       |2488358900|
|2025-02-01 00:00:00|13082       |2258084500|
|2025-03-01 00:00:00|14845       |2674323900|
|2025-04-01 00:00:00|14690       |2517584800|
|2025-05-01 00:00:00|15393       |2687288800|
|2025-06-01 00:00:00|15374       |2711347100|
|2025-07-01 00:00:00|16096       |2781402600|
|2025-08-01 00:00:00|16253       |2828046300|
|2025-09-01 00:00:00|16180       |2878092500|
|2025-10-01 00:00:00|17019       |2975726900|
|2025-11-01 00:00:00|17023       |2974285100|
|2025-12-01 00:00:00|26144       |45869521

*Votre réponse :*

…

### D3 — Panier moyen par canal
`ROUND(AVG(montant_fcfa), 0)` — mobile ou web, qui dépense le plus par commande ?

In [26]:
# === D3 - Panier moyen par canal ===
print("=" * 60)
print("D3 - Panier moyen par canal")
print("=" * 60)

panier = spark.sql("""
    SELECT canal,
           COUNT(*)                    AS nb_commandes,
           SUM(montant_fcfa)           AS ca_fcfa,
           ROUND(AVG(montant_fcfa), 0) AS panier_moyen_fcfa
    FROM commandes_clean
    GROUP BY canal
    ORDER BY ca_fcfa DESC
""")
panier.show()

# Extraction des valeurs
resultats_panier = spark.sql("""
    SELECT 
        ROUND(AVG(CASE WHEN canal = 'mobile_app' THEN montant_fcfa END), 0) AS panier_mobile,
        ROUND(AVG(CASE WHEN canal = 'web' THEN montant_fcfa END), 0) AS panier_web,
        ROUND(AVG(montant_fcfa), 0) AS panier_global
    FROM commandes_clean
""").first()

releves["D3_panier_moyen_mobile"] = resultats_panier["panier_mobile"]
releves["D3_panier_moyen_web"] = resultats_panier["panier_web"]
releves["D3_panier_moyen_global"] = resultats_panier["panier_global"]

print("\n" + "-" * 60)
print("📊 RELEVÉ D3 - Panier moyen :")
print(f"  Mobile App  : {releves['D3_panier_moyen_mobile']:,.0f} FCFA")
print(f"  Web         : {releves['D3_panier_moyen_web']:,.0f} FCFA")
print(f"  Global      : {releves['D3_panier_moyen_global']:,.0f} FCFA")
print("-" * 60)

# Interprétation
print("\n💡 INTERPRÉTATION :")
if releves['D3_panier_moyen_mobile'] > releves['D3_panier_moyen_web']:
    print(f"  ✅ Les clients mobile dépensent en moyenne {releves['D3_panier_moyen_mobile'] - releves['D3_panier_moyen_web']:,.0f} FCFA de plus que les clients web.")
    print(f"  Soit {(releves['D3_panier_moyen_mobile']/releves['D3_panier_moyen_web'] - 1) * 100:.1f}% de panier moyen en plus.")
elif releves['D3_panier_moyen_web'] > releves['D3_panier_moyen_mobile']:
    print(f"  ✅ Les clients web dépensent en moyenne {releves['D3_panier_moyen_web'] - releves['D3_panier_moyen_mobile']:,.0f} FCFA de plus que les clients mobile.")
else:
    print("  ℹ️ Les deux canaux ont un panier moyen équivalent.")

D3 - Panier moyen par canal
+----------+------------+-----------+-----------------+
|     canal|nb_commandes|    ca_fcfa|panier_moyen_fcfa|
+----------+------------+-----------+-----------------+
|mobile_app|      324565|56702257000|         174702.0|
|       web|      175435|30749408200|         175275.0|
+----------+------------+-----------+-----------------+


------------------------------------------------------------
📊 RELEVÉ D3 - Panier moyen :
  Mobile App  : 174,702 FCFA
  Web         : 175,275 FCFA
  Global      : 174,903 FCFA
------------------------------------------------------------

💡 INTERPRÉTATION :
  ✅ Les clients web dépensent en moyenne 573 FCFA de plus que les clients mobile.


*Votre réponse :*

…

### D4 — Top clients, avec HAVING
Top 10 des clients par CA (`GROUP BY customer_id`), en ne gardant que les clients dépassant **1 000 000 FCFA** de CA cumulé. Un client sort-il du lot ?

In [27]:
# === D4 - Top 10 des clients par CA ===
print("=" * 60)
print("D4 - Top 10 des clients par CA")
print("=" * 60)

spark.sql("""
    SELECT customer_id,
           COUNT(*)          AS nb_commandes,
           SUM(montant_fcfa) AS ca_fcfa
    FROM commandes_clean
    GROUP BY customer_id
    HAVING SUM(montant_fcfa) > 1000000
    ORDER BY ca_fcfa DESC
    LIMIT 10
""").show(truncate=False)

# === Extraction du client avec le plus gros CA ===
top_client = spark.sql("""
    SELECT customer_id,
           COUNT(*)          AS nb_commandes,
           SUM(montant_fcfa) AS ca_fcfa
    FROM commandes_clean
    GROUP BY customer_id
    ORDER BY ca_fcfa DESC
    LIMIT 1
""").first()

releves["D4_top_client_id"] = top_client["customer_id"]
releves["D4_top_client_ca"] = top_client["ca_fcfa"]
releves["D4_top_client_nb_commandes"] = top_client["nb_commandes"]

print("\n" + "-" * 60)
print("📊 RELEVÉ D4 - Top client :")
print(f"  Client ID            : {releves['D4_top_client_id']}")
print(f"  CA total             : {releves['D4_top_client_ca']:,.0f} FCFA")
print(f"  Nombre de commandes  : {releves['D4_top_client_nb_commandes']:,}")
print("-" * 60)

D4 - Top 10 des clients par CA
+-----------+------------+----------+
|customer_id|nb_commandes|ca_fcfa   |
+-----------+------------+----------+
|C000001    |27482       |4847721800|
|C000002    |1605        |273206000 |
|C000003    |1325        |251194200 |
|C000005    |1016        |176441300 |
|C000004    |1074        |176140200 |
|C000007    |826         |147481400 |
|C000006    |847         |146904200 |
|C000009    |702         |122985000 |
|C000008    |711         |120414300 |
|C000011    |580         |104503800 |
+-----------+------------+----------+


------------------------------------------------------------
📊 RELEVÉ D4 - Top client :
  Client ID            : C000001
  CA total             : 4,847,721,800 FCFA
  Nombre de commandes  : 27,482
------------------------------------------------------------


*Votre réponse :*

…

### D5 — Paiements par méthode
Nombre et pourcentage par méthode. Quelle part totale pour le **mobile money** (Orange Money + Wave + Free Money) ?

In [28]:
# === D5 - Répartition des paiements par méthode ===
print("=" * 60)
print("D5 - Répartition des paiements par méthode")
print("=" * 60)

spark.sql("""
    SELECT methode,
           COUNT(*) AS nb,
           ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS part_pct
    FROM paiements
    GROUP BY methode
    ORDER BY nb DESC
""").show(truncate=False)

# Calcul de la part du mobile money (Orange Money + Wave + Free Money)
part_mobile_money = spark.sql("""
    SELECT ROUND(
        100 * SUM(CASE WHEN methode IN ('Orange Money', 'Wave', 'Free Money') THEN 1 ELSE 0 END) / COUNT(*), 
        1
    ) AS part_mobile_money_pct
    FROM paiements
""").first()

releves["D5_part_mobile_money_pct"] = part_mobile_money["part_mobile_money_pct"]

print(f"\n📊 Part du mobile money : {releves['D5_part_mobile_money_pct']:.1f}%")

D5 - Répartition des paiements par méthode
+-----------------------+------+--------+
|methode                |nb    |part_pct|
+-----------------------+------+--------+
|Orange Money           |155931|34.9    |
|Wave                   |133964|30.0    |
|Paiement à la livraison|98664 |22.1    |
|Carte bancaire         |35574 |8.0     |
|Free Money             |22237 |5.0     |
+-----------------------+------+--------+


📊 Part du mobile money : 69.9%


*Votre réponse :*

…

## Partie E — SQL ou API ? `explain()` tranche (20 min)

### E1 — D3 en API DataFrame
Réécrivez le panier moyen par canal avec `groupBy().agg()`. Les chiffres doivent être **identiques**.

In [29]:
from pyspark.sql import functions as F

# === E1 - Panier moyen par canal avec l'API DataFrame ===
print("=" * 60)
print("E1 - Panier moyen par canal (API DataFrame)")
print("=" * 60)

commandes_clean_df = spark.table("commandes_clean")

panier_api = (commandes_clean_df
    .groupBy("canal")
    .agg(
        F.count("*").alias("nb_commandes"),
        F.sum("montant_fcfa").alias("ca_fcfa"),
        F.round(F.avg("montant_fcfa"), 0).alias("panier_moyen_fcfa")
    )
    .orderBy(F.col("ca_fcfa").desc()))

panier_api.show()

E1 - Panier moyen par canal (API DataFrame)
+----------+------------+-----------+-----------------+
|     canal|nb_commandes|    ca_fcfa|panier_moyen_fcfa|
+----------+------------+-----------+-----------------+
|mobile_app|      324565|56702257000|         174702.0|
|       web|      175435|30749408200|         175275.0|
+----------+------------+-----------+-----------------+



### E2 — B4 en API
La même requête « livrées / mobile / décembre 2025 », version `filter`.

In [30]:
# === E2 - Version API pour B4 (commandes livrées mobile_app en décembre 2025) ===
print("=" * 60)
print("E2 - Commandes livrées mobile_app en décembre 2025 (API)")
print("=" * 60)

nb_api = (spark.table("commandes")
    .filter(
        (F.col("canal") == "mobile_app") &
        (F.col("statut") == "livrée") &
        (F.col("date_commande").between("2025-12-01", "2025-12-31"))
    )
    .count())

print(f"Nombre de commandes : {nb_api:,}")
releves["B4_api_nb_commandes"] = nb_api

E2 - Commandes livrées mobile_app en décembre 2025 (API)
Nombre de commandes : 16,415


### E3 — Comparer les plans
Affichez le plan physique de la version SQL de D3 et de `panier_api`.

In [31]:
# === E3 - Comparaison des plans d'exécution SQL vs API ===
print("=" * 60)
print("E3 - Comparaison des plans d'exécution (SQL vs API)")
print("=" * 60)

# Version SQL
panier_sql = spark.sql("""
    SELECT canal, 
           COUNT(*) AS nb_commandes,
           SUM(montant_fcfa) AS ca_fcfa,
           ROUND(AVG(montant_fcfa), 0) AS panier_moyen_fcfa
    FROM commandes_clean
    GROUP BY canal
    ORDER BY ca_fcfa DESC
""")

# Version API
from pyspark.sql import functions as F

panier_api = (spark.table("commandes_clean")
    .groupBy("canal")
    .agg(
        F.count("*").alias("nb_commandes"),
        F.sum("montant_fcfa").alias("ca_fcfa"),
        F.round(F.avg("montant_fcfa"), 0).alias("panier_moyen_fcfa")
    )
    .orderBy(F.col("ca_fcfa").desc()))

# Comparaison des plans
print("\n" + "=" * 60)
print("📊 PLAN D'EXÉCUTION - SQL")
print("=" * 60)
panier_sql.explain()

print("\n" + "=" * 60)
print("📊 PLAN D'EXÉCUTION - API")
print("=" * 60)
panier_api.explain()

# Comparaison détaillée avec explain(True)
print("\n" + "=" * 60)
print("📊 PLAN DÉTAILLÉ - SQL (True)")
print("=" * 60)
panier_sql.explain(True)

print("\n" + "=" * 60)
print("📊 PLAN DÉTAILLÉ - API (True)")
print("=" * 60)
panier_api.explain(True)

E3 - Comparaison des plans d'exécution (SQL vs API)

📊 PLAN D'EXÉCUTION - SQL
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [ca_fcfa#821L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(ca_fcfa#821L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=1679]
      +- HashAggregate(keys=[canal#89], functions=[count(1), sum(montant_fcfa#832L), avg(montant_fcfa#832L)])
         +- Exchange hashpartitioning(canal#89, 200), ENSURE_REQUIREMENTS, [plan_id=1676]
            +- HashAggregate(keys=[canal#89], functions=[partial_count(1), partial_sum(montant_fcfa#832L), partial_avg(montant_fcfa#832L)])
               +- Project [canal#89, cast(regexp_replace(trim(montant_total_fcfa#91, None), [^0-9], , 1) as bigint) AS montant_fcfa#832L]
                  +- FileScan csv [canal#89,montant_total_fcfa#91] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/bigdata-ISI-2026-rene-legrand-mountata/data/orders.csv], PartitionFilters: [

**Question E** — Les plans physiques sont-ils identiques ? Où voit-on le filtre poussé vers la lecture (`PushedFilters` / `Filter` près du `FileScan`) ? Que concluez-vous sur le choix SQL vs API ? (3 phrases)

*Votre réponse :*
# === Question E - Réponse ===


question_E_reponse = """
Les plans physiques sont-ils identiques ? 

Oui, les plans physiques sont généralement identiques car Spark utilise le même 
optimiseur Catalyst pour les requêtes SQL et les opérations DataFrame. Les deux 
approches produisent le même plan logique optimisé et le même plan physique.

Où voit-on le filtre poussé vers la lecture du fichier ?

Le filtre apparaît dans la phase de lecture (FileScan) sous forme de prédicats 
poussés (pushdown predicates). Spark applique les filtres dès la lecture des 
fichiers pour réduire le volume de données à traiter, ce qui améliore 
considérablement les performances.
"""

print(question_E_reponse)
…

## Partie F — Discussion guidée (10 min, en groupes)

1. Le CA naïf de C1 était faux **en silence**. Dans une vraie entreprise, qui s'en serait aperçu, quand, et à quel coût ? Proposez **deux garde-fous techniques**.
2. `commandes_clean` est une vue temporaire : que se passe-t-il demain matin au redémarrage du notebook ? Est-ce acceptable en production ? (indice : séances 6-7)
3. La direction veut le CA par **ville du client** : quelle information manque à `commandes_clean` seule, et comment l'obtiendrez-vous en séance 5 ?

# ============================================
# PARTIE F — Discussion guidée et Quiz (Version concise)
# ============================================

print("=" * 70)
print("PARTIE F — DISCUSSION GUIDÉE ET QUIZ")
print("=" * 70)

# ------------------------------------------------------------------
# F.1 — Discussion guidée (Réponses courtes)
# ------------------------------------------------------------------
print("\n" + "=" * 60)
print("F.1 — DISCUSSION GUIDÉE")
print("=" * 60)

print("""
1. CA faux en silence :
   - Qui ? DAF, contrôleur de gestion, équipe data
   - Quand ? Revues mensuelles/trimestrielles
   - Coût ? Financier, réputation, temps de correction

2. Garde-fous techniques :
   a) Validation à l'ingestion (tests de qualité)
   b) Dashboards de monitoring avec alertes

3. Vue temporaire :
   - Disparaît au redémarrage
   - Pas acceptable en production
   - Solution : tables persistantes ou pipelines ETL

4. CA par ville :
   - Manque la ville (dans clients)
   - Solution : JOIN commandes_clean + clients sur customer_id
""")

# ------------------------------------------------------------------
# F.2 — Quiz éclair 
# ------------------------------------------------------------------
print("\n" + "=" * 60)
print("F.2 — QUIZ ÉCLAIR")
print("=" * 60)

print("""
1. spark.sql() retourne : DataFrame

2. createOrReplaceTempView :
   - Ne copie PAS les données
   - Portée : SparkSession actuel

3. WHERE email = NULL ne renvoie rien car :
   - NULL n'est égal à rien, même pas NULL
   - Utiliser IS NULL / IS NOT NULL

4. SUM sur string polluée :
   - Spark récent : ERREUR
   - Spark ancien : résultat FAUX
   - Dangereux car décisions erronées

5. WHERE vs HAVING :
   - WHERE : filtre les lignes (avant GROUP BY)
   - HAVING : filtre les groupes (après GROUP BY)
""")

# Score
print("\n" + "-" * 60)
print("📝 MON SCORE AU QUIZ : ___ / 5")
print("-" * 60)


# ------------------------------------------------------------------
# Relevés finaux
# ------------------------------------------------------------------
print("\n" + "=" * 60)
print("TABLEAU DE RELEVÉS — TP4")
print("=" * 60)

for k, v in releves.items():
    if isinstance(v, float):
    
        print(f"{k:32s} : {v:.2f}")
    else:
        print(f"{k:32s} : {v}")

manquants = [k for k, v in releves.items() if v is None or v == "" or v == "..."]
print("\n" + "-" * 60)
if manquants:
    print(f"⚠️ Relevés manquants : {len(manquants)}")
    for k in manquants:
        print(f"  - {k}")
else:
    print("✅ Tous les relevés sont complets — bravo !")
print("-" * 60)

# ------------------------------------------------------------------
# Résumé des indicateurs clés
# ------------------------------------------------------------------
print("\n" + "=" * 60)
print("📊 INDICATEURS CLÉS — RÉSUMÉ")
print("=" * 60)

print(f"  CA total nettoyé            : {releves.get('C4_ca_nettoye', 0):>15,.0f} FCFA")
print(f"  Écart CA (C1 vs C4)         : {releves.get('C5_ecart_fcfa', 0):>15,.0f} FCFA")
print(f"  Écart en %                  : {releves.get('C5_ecart_pct', 0):>15.2f}%")
print(f"  Valeurs polluées            : {releves.get('C2_nb_valeurs_polluees', 0):>15,}")
print(f"  NULL après nettoyage        : {releves.get('C4_nb_null', 0):>15}")
print(f"  Part CA livrée              : {releves.get('D1_part_ca_livree_pct', 0):>15.2f}%")
print(f"  Mois record                 : {releves.get('D2_mois_record', 'N/A'):>15}")
print(f"  Panier moyen mobile         : {releves.get('D3_panier_moyen_mobile', 0):>15,.0f} FCFA")
print(f"  Top client ID               : {releves.get('D4_top_client_id', 'N/A'):>15}")
print(f"  Top client CA               : {releves.get('D4_top_client_ca', 0):>15,.0f} FCFA")
print(f"  Part mobile money           : {releves.get('D5_part_mobile_money_pct', 0):>15.1f}%")
print("=" * 60)

print("\n✅ TP4 terminé avec succès !")

In [32]:
print("\n" + "=" * 60)
print("📦 LIVRABLE")
print("=" * 60)

print("""
Depuis la racine du dépôt :

  git status                       # vérifier que data/ n'apparaît PAS
  git add notebooks/TP4_spark_sql.ipynb
  git commit -m "TP4 : requetes SQL et nettoyage FCFA"
  git push

Checklist :
  □ Notebook exécuté (Restart & Run All)
  □ nb_null = 0 dans C4
  □ Écart CA en FCFA et %
  □ Interprétations sous chaque D
  □ Aucun relevé manquant
  □ Données non commitées
""")
print("=" * 60)


📦 LIVRABLE

Depuis la racine du dépôt :

  git status                       # vérifier que data/ n'apparaît PAS
  git add notebooks/TP4_spark_sql.ipynb
  git commit -m "TP4 : requetes SQL et nettoyage FCFA"
  git push

Checklist :
  □ Notebook exécuté (Restart & Run All)
  □ nb_null = 0 dans C4
  □ Écart CA en FCFA et %
  □ Interprétations sous chaque D
  □ Aucun relevé manquant
  □ Données non commitées



### Pousser le livrable

Depuis la racine de votre dépôt :
```
git status                       # verifier que data/ n'apparait PAS
git add notebooks/TP4_spark_sql.ipynb
git commit -m "TP4 : requetes SQL et nettoyage FCFA"
git push
```

**Checklist finale**
- [ ] Notebook exécuté de bout en bout (Restart & Run All), sorties visibles ;
- [ ] `nb_null = 0` dans la validation C4 ;
- [ ] Écart CA naïf / nettoyé relevé en FCFA **et** en % ;
- [ ] Une phrase d'interprétation sous chaque indicateur de la partie D ;
- [ ] Aucun relevé manquant dans la cellule ci-dessus ;
- [ ] Données non commitées.

*Séance 5 : les jointures — lecture préalable : Damji et al., Learning Spark 2e éd., chapitre 5.*